# AquaCubeAI — training and evaluation (20 × 20)

This notebook orchestrates the paper workflow. Reusable code lives in `aquacubeai/`.
Obtain the dataset from the authors and follow `data/README.md` before running it.

The model learns from eight-band patch means. Targets are filtered to
**0 ≤ TUR < 140 FNU** and scaled using the training subset only. Groups combine
a CSV acquisition identifier and a 15-km spatial cell. The original source-list
order and multiplicity are retained.

Run all cells in order. Full training includes the original default run followed
by the architecture-search stage and may take substantial time. New artifacts
are written under `outputs/`; supplied reference models in `models/` are preserved.

In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

# Works when Jupyter starts in the repository root or in notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "aquacubeai").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the AquaCubeAI repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## 1. Configuration

In [ ]:
from aquacubeai.config import Config, data_directory, set_seed
from aquacubeai.data import (
    source_pairs,
    load_feature_dataset,
    build_group_split,
    prepare_splits,
    get_dataset_from_csv,
    patch_mean_features,
    apply_minmax_np,
    load_norm_params,
    clean_tensor,
)
from aquacubeai.training import train_mlp_regressor, architecture_search, select_best
from aquacubeai.evaluation import (
    evaluate_checkpoint,
    load_best_model,
    predict_loader,
    denorm_if_needed,
    compute_metrics_np,
    threshold_metrics,
)
from aquacubeai.models import save_spatial_model
from aquacubeai.plotting import (
    plot_split_histograms,
    plot_learning_curves,
    plot_regression,
    plot_distributions,
)

# Optional: set this before running if the data are stored outside this repository.
# os.environ["AQUACUBEAI_DATA_DIR"] = r"/path/to/supplied/data"
DATA_ROOT = data_directory()
RUN_DIR = ROOT / "outputs" / "training"
RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg = Config(
    norm_params_path=str(RUN_DIR / "norm_params_tur.npz"),
    best_model_path=str(RUN_DIR / "best_model.pth"),
)
cfg.verbose = True  # Training progress every 25 epochs; set False for quiet execution.
arch_candidates = [(512, 512, 256)]  # Active candidate in the reference script.
SHOW_DISTRIBUTIONS = False
set_seed(cfg.seed)
print(f"Device: {cfg.device} | Seed: {cfg.seed} | Data: {DATA_ROOT}")


## 2. Data and group-aware split

Patch descriptors are computed one acquisition at a time to avoid retaining the
complete collection of raw patches in memory. The float32 mean operation,
matching rule (nearest filename coordinates within 2.5 m), target filter and
source order are unchanged. Groups are disjoint between subsets; grouping does
not reserve complete geographical sites or acquisition dates for one subset.

In [ ]:
sources = source_pairs(data_root=DATA_ROOT)
features, targets, groups = load_feature_dataset(sources, cfg)
train_idx, val_idx, test_idx, split_score = build_group_split(features, targets, groups, cfg)
indices = (train_idx, val_idx, test_idx)
split_table = pd.DataFrame(
    [
        {
            "Subset": name,
            "Samples": len(idx),
            "Fraction": len(idx) / len(targets),
            "Mean FNU": targets[idx].mean(),
            "Std FNU": targets[idx].std(),
            "Min FNU": targets[idx].min(),
            "Max FNU": targets[idx].max(),
        }
        for name, idx in zip(("Training", "Validation", "Test"), indices)
    ]
)
display(split_table)
print(f"Groups: {len(set(groups))} | Split score: {split_score:.6f}")
np.savez_compressed(RUN_DIR / "split_indices.npz", train=train_idx, val=val_idx, test=test_idx)
fig = plot_split_histograms(targets, indices)
plt.show()


## 3. Training-only target scaling

Input descriptors receive no additional normalization. PCA is disabled in the
paper configuration. If enabled for a separate MLP experiment, it is fitted on
training descriptors and the dense-model export is disabled.

In [ ]:
tensors, pca = prepare_splits(features, targets, indices, cfg)
display(
    pd.DataFrame(
        [
            {"Subset": name, "Feature shape": tuple(x.shape), "Target shape": tuple(y.shape)}
            for name, (x, y) in tensors.items()
        ]
    )
)


## 4. Default training run

Adam, Huber loss (δ = 1), gradient clipping (0.5), the original learning-rate
scheduler and early stopping are retained. The seed is reset before each run.
Validation loss selects the checkpoint; the training loop averages batch losses
in the same way as the original implementation.

In [ ]:
best_val, best_epoch, train_history, val_history = train_mlp_regressor(
    *tensors["train"], *tensors["val"], cfg.hidden_layers, cfg, cfg.best_model_path
)
default_output = evaluate_checkpoint(cfg.best_model_path, cfg.hidden_layers, tensors, cfg)
print(f"Best validation loss: {best_val:.6f} | Best epoch (zero-based): {best_epoch}")
display(pd.DataFrame(default_output["metrics"]).T)
fig = plot_learning_curves(train_history, val_history)
plt.show()


## 5. Architecture search and final checkpoint

This stage retains the original caps of 800 epochs and patience 120.
Each candidate receives a separate checkpoint. Final selection uses validation
RMSE in FNU, as in the original post-hoc selection. Test metrics are reported
after training and are not the selection criterion.

In [ ]:
results = architecture_search(arch_candidates, tensors, cfg, RUN_DIR)
selected = select_best(results)
selected_output = selected["evaluation"]
best_checkpoint = selected["checkpoint"]
best_architecture = selected["arch"]
display(
    pd.DataFrame(
        [
            {
                "Architecture": r["arch"],
                "Best epoch": r["best_epoch"],
                "Best validation loss": r["best_val"],
                "Validation RMSE": r["evaluation"]["metrics"]["val"]["rmse"],
                "Test RMSE": r["evaluation"]["metrics"]["test"]["rmse"],
            }
            for r in results
        ]
    )
)
display(pd.DataFrame(selected_output["metrics"]).T)
fig = plot_regression(selected_output["preds"])
fig.savefig(RUN_DIR / "regression_subsets.png", dpi=200)
plt.show()
if SHOW_DISTRIBUTIONS:
    fig = plot_distributions(selected_output["preds"])
    plt.show()


## 6. Turbidity ranges and threshold exceedance

In [ ]:
# Ranges are cumulative and overlap; their sample counts must not be summed.
range_rows = [
    {"Subset": name, "Range FNU": label, **metrics}
    for name, ranges in selected_output["metrics_by_range"].items()
    for label, metrics in ranges.items()
]
display(pd.DataFrame(range_rows))
threshold_table = pd.DataFrame(
    {
        name: threshold_metrics(*selected_output["preds"][name], threshold=100.0)
        for name in ("val", "test")
    }
).T
display(threshold_table)


## 7. Spatial checkpoint and ONNX export

The trained MLP weights are converted into 1 × 1 convolutions. The exported
model accepts `[1, 8, 20, 20]` and returns `[1, 1, 20, 20]` in normalized target
units. Apply the accompanying training scaling parameters to obtain FNU.
This cell writes new artifacts under the run directory, not `models/`.

In [ ]:
import shutil

if not cfg.pca_enable:
    model_area = save_spatial_model(
        best_checkpoint,
        best_architecture,
        RUN_DIR,
        patch_hw=cfg.patch_hw,
        pca_enabled=cfg.pca_enable,
    )
    print("Saved spatial checkpoint and ONNX model:", RUN_DIR)
else:
    print("Dense export skipped: PCA is enabled.")
report = {
    "config": cfg.__dict__,
    "selected_architecture": best_architecture,
    "selected_checkpoint": Path(best_checkpoint).name,
    "selection_criterion": "validation RMSE",
    "best_epoch": selected["best_epoch"],
    "split_score": split_score,
    "metrics": selected_output["metrics"],
    "metrics_by_range": selected_output["metrics_by_range"],
}
(RUN_DIR / "training_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
threshold_table.to_csv(RUN_DIR / "threshold_metrics.csv")
pd.DataFrame(range_rows).to_csv(RUN_DIR / "metrics_by_range.csv", index=False)


## 8. External evaluation

This evaluates the MLP on patch-averaged spectra, using the same preprocessing
and training-only target scaling. These scalar results differ from averaging
a dense map: in general, `f(mean spectra)` and `mean(f(pixel spectra))` differ.
The CMEMS HR-OC targets are reference products, not direct in situ measurements.

In [ ]:
patch_dir, csv_path = source_pairs(data_root=DATA_ROOT, subset="external")[0]
external_patches, external_y = get_dataset_from_csv(patch_dir, csv_path, verbose=cfg.verbose)
external_y = np.asarray(external_y, dtype=np.float32).reshape(-1, 1)
keep = (
    np.isfinite(external_y.ravel())
    & (external_y.ravel() >= cfg.tur_min)
    & (external_y.ravel() < cfg.tur_max)
)
external_x = patch_mean_features([patch for patch, ok in zip(external_patches, keep) if ok])
external_y = external_y[keep]
if pca is not None:
    external_x = pca.transform(external_x).astype(np.float32)
external_target = (
    apply_minmax_np(external_y, load_norm_params(cfg.norm_params_path))
    if cfg.normalization_enable
    else external_y
)
model = load_best_model(external_x.shape[1], 1, best_architecture, best_checkpoint, cfg.device)
predicted_t, reference_t = predict_loader(
    model,
    torch.tensor(external_x, dtype=torch.float32),
    torch.tensor(external_target, dtype=torch.float32),
)
external_true, external_pred = denorm_if_needed(reference_t, predicted_t, cfg)
external_metrics = compute_metrics_np(external_true, external_pred)
display(pd.DataFrame([{"Samples": len(external_true), **external_metrics}]))
slope, intercept = np.polyfit(external_true.ravel(), external_pred.ravel(), 1)
print(f"Fit: predicted TUR = {slope:.8f} * reference TUR {intercept:+.8f}")
print(f"Training seed: {cfg.seed} | Checkpoint: {Path(best_checkpoint).name}")
fig = plot_regression({"external test": (external_true, external_pred)})
fig.savefig(RUN_DIR / "external_test.png", dpi=200)
plt.show()
(RUN_DIR / "external_metrics.json").write_text(
    json.dumps(
        {
            "source": Path(csv_path).name,
            "n": len(external_true),
            "metrics": external_metrics,
            "slope": float(slope),
            "intercept_fnu": float(intercept),
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
